In [0]:


INTERFACE_NAME = "PLAN_SHIPMENTS"

for name, label in [
    ("batch_control_key", "Batch Control Key"),
    ("silver_table_name", "Silver Table"),
    ("partition_column", "Partition Column"),
    ("temp_view", "Temp View Name"),
    ("source_system", "Source System"),
    ("bronze_table_name", "Bronze Table"),
]:
    dbutils.widgets.text(name, "", label)

batch_control_key = dbutils.widgets.get("batch_control_key").strip()
silver_table_name = dbutils.widgets.get("silver_table_name").strip()
partition_column = dbutils.widgets.get("partition_column").strip() or None
temp_view = dbutils.widgets.get("temp_view").strip()
source_system = dbutils.widgets.get("source_system").strip()
bronze_table_name = dbutils.widgets.get("bronze_table_name").strip()

assert batch_control_key, "batch_control_key parameter is required"
assert silver_table_name, "silver_table_name parameter is required"
assert temp_view, "temp_view parameter is required"

print(f" Silver Custom Notebook : {INTERFACE_NAME}")
print(f" Reading framework view : global_temp.{temp_view}")
print(f" Target silver table    : {silver_table_name}")

# The framework has already parsed XML, applied XSD typing and flattened the payload.
df = spark.table(f"global_temp.{temp_view}")
print(f" Framework columns     : {len(df.columns)}")


In [0]:

business_columns = [
    "companyCode", "facility", "warehouse", "planShipmentNumber",
    "shipTo", "shipToType", "shipmentStatus", "dateShipped",
    "deliveryId", "orderType", "creationDate", "orderNumber",
    "orderReference", "custOrderRef", "planShipmentLineNumber",
    "itemCode", "hostItem", "itemOwner", "itemFacility", "itemWarehouse",
    "itemPackCode", "specialApportionment", "substitutionFlag",
    "orderLine", "constrainingStatus",
    "orderedQty", "acceptedQty", "adjustedQty", "remainingQty",
    "releasedQuantity", "pickedQty", "openToPickQty", "sentQty",
    "lineStatus", "lineSource",
    "nonDeliveryCode1", "nonDeliveryDescription1",
    "nonDeliveryCode2", "nonDeliveryDescription2",
    "nonDeliveryCode3", "nonDeliveryDescription3",
    "partOfItem", "psTimeStamp"
]

# These fields are needed to identify a plan shipment line and to build _record_guid.
required_key_columns = [
    "facility", "warehouse", "itemCode",
    "planShipmentNumber", "planShipmentLineNumber",
    "orderNumber", "shipTo", "creationDate", "psTimeStamp"
]

# Four-stage DQ guard from UTILS: missing-required, add-optional-NULL,
# all-NULL check, NULL-key check.
df = validate_silver_input(
    df,
    business_columns,
    required_key_columns,
    interface_name=INTERFACE_NAME,
)

print(f" \u2713 Validated framework input for {INTERFACE_NAME}")

In [0]:


# IMPORTANT: no business type casting here. The framework already applies XSD types.
# Schema alignment to the target table happens inside merge_to_silver().
rename_map = {
    "cpyCd": "cpy_cd",
    "facl": "facl",
    "whse": "whse",
    "pr": "pr",
    "prReference": "pr_reference",
    "asnShipment": "asn_shipment",
    "itemOwner": "item_owner",
    "prType": "pr_type",
    "schdDate": "schd_date",
    "schdTime": "schd_time",
    "schdDoor": "schd_door",
    "prStat": "pr_stat",
    "fromTrader": "from_trader",
    "fromTraderType": "from_trader_type",
    "deliveryMethod": "delivery_method",
    "loadType": "load_type",
    "handRcvFlag": "hand_rcv_flag",
    "prCarrier": "pr_carrier",
    "receiveDate": "receive_date",
    # ... mapped columns continue down to:
    "planShipment": "plan_shipment",
    "lotCode": "lot_code",
    "acknowledgeQty": "acknowledge_qty",
    "erpItemBasePack": "erp_item_base_pack",
    "erpItemBaseQty": "erp_item_base_qty",
    "rcptExceptionQty1": "rcpt_exception_qty1",
    "rcptExceptionQty2": "rcpt_exception_qty2",
    "prTimeStamp": "pr_timestamp",
}

# Rename business columns + carry FRAMEWORK_METADATA_COLUMNS from UTILS.
silver_final_df = rename_with_metadata(df, rename_map)

In [0]:

silver_final_df = create_record_guid(silver_final_df, [
    F.col("cpy_cd").cast("string"),
    F.col("facl").cast("string"),
    F.col("whse").cast("string"),
    F.col("item_code").cast("string"),
    F.col("pr_reference").cast("string"),
    F.col("pr_ref_line").cast("string"),
    F.col("from_trader").cast("string"),
    F.date_format(F.col("pr_crt_date"), "yyyyMMdd"),
    F.date_format(F.col("pr_timestamp"), "yyyyMMddHHmmss")
], '_business_event_guid')


In [0]:
# silver_final_df = silver_final_df.withColumn(
#     "_business_event_guid",
#     F.md5(
#         F.concat_ws(
#             "~",
#             F.col("facility").cast("string"),
#             F.col("warehouse").cast("string"),
#             F.col("item_code").cast("string"),
#             F.col("plan_shipment_number").cast("string"),
#             F.col("plan_shipment_line_number").cast("string"),
#             F.col("order_number").cast("string"),
#             F.col("ship_to").cast("string"),
#             F.date_format(F.col("creation_date"), "yyyyMMdd"),
#             F.date_format(F.col("ps_time_stamp"), "yyyyMMddHHmmss"),
#         )
#     ),
# )

# silver_final_df = silver_final_df.select(
#     "_record_guid", "_business_event_guid", *[c for c in silver_final_df.columns if c not in ["_record_guid", "_business_event_guid"]]
# )


In [0]:

records_processed = write_to_silver(
    silver_final_df,
    silver_table_name,
    batch_control_key,
    partition_column=partition_column,
)

print(f" Silver write complete : {records_processed} records")
dbutils.notebook.exit(json.dumps({"records_processed": records_processed}))